<a href="https://colab.research.google.com/github/Moharram-Khaled/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*


The playbook prioritizes content pages using the validated model output and the baseline action score as decision-support signals.

The primary action is:

- `review_ranking_opportunity` — send the page for human review before making any content change.

Reason codes:

- `high_visibility_low_position` — the page has meaningful search visibility but a relatively weak average position.
- `model_predicted_improvement` — the validated model predicts a directional likelihood of future position improvement.

These signals are prioritization aids, not automatic recommendations.

In [5]:
import os
import pandas as pd
from google.colab import userdata
from huggingface_hub import hf_hub_download

HF_TOKEN = userdata.get("HF_TOKEN")

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march = pd.read_parquet(march_path)

print("Rows:", len(march))

Rows: 9841378


In [6]:
# Keep only rows with the signals required by the playbook
playbook = march[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_avg_position",
        "gsc_clicks",
        "gsc_data_available"
    ]
].copy()

# Percentile ranks
playbook["impressions_rank"] = playbook["gsc_impressions"].rank(
    pct=True,
    method="average"
)

playbook["position_rank"] = playbook["gsc_avg_position"].rank(
    pct=True,
    method="average"
)

# Baseline action score
playbook["action_score"] = (
    0.5 * playbook["impressions_rank"]
    + 0.5 * playbook["position_rank"]
)

playbook["reason_code"] = "high_visibility_low_position"
playbook["action"] = "review_ranking_opportunity"

# Ranked queue
queue = playbook[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_avg_position",
        "gsc_clicks",
        "action_score",
        "reason_code",
        "action"
    ]
].sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

queue.insert(0, "rank", range(1, len(queue) + 1))

print("Queue size:", len(queue))
display(queue.head(20))

Queue size: 9841378


,rank,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_avg_position,gsc_clicks,action_score,reason_code,action
0,1,client_20259bd6705d81d4,content_1c14838b50c55c9f,2026-03-25,1190,134.111765,0,0.998694,high_visibility_low_position,review_ranking_opportunity
1,2,client_20259bd6705d81d4,content_f4bd56290b265782,2026-03-25,1146,105.913613,0,0.998492,high_visibility_low_position,review_ranking_opportunity
2,3,client_20259bd6705d81d4,content_b6259d89f57d5ebc,2026-03-25,986,96.248479,3,0.996996,high_visibility_low_position,review_ranking_opportunity
3,4,client_20259bd6705d81d4,content_6c67f84d8b90e7c5,2026-03-25,689,118.474601,0,0.996719,high_visibility_low_position,review_ranking_opportunity
4,5,client_20259bd6705d81d4,content_dbdef2d26fa09819,2026-03-25,667,117.287856,0,0.996541,high_visibility_low_position,review_ranking_opportunity
5,6,client_20259bd6705d81d4,content_4c8f60c8875f8cc6,2026-03-25,640,99.345313,4,0.996198,high_visibility_low_position,review_ranking_opportunity
6,7,client_20259bd6705d81d4,content_3896bdc4b2b82656,2026-03-25,617,121.458671,0,0.996122,high_visibility_low_position,review_ranking_opportunity
7,8,client_20259bd6705d81d4,content_4c82c93a3ee8cba8,2026-03-25,560,149.058929,0,0.995586,high_visibility_low_position,review_ranking_opportunity
8,9,client_23a62021009f63c4,content_6530fa9d297c46eb,2026-03-31,5364,89.848248,0,0.995552,high_visibility_low_position,review_ranking_opportunity
9,10,client_20259bd6705d81d4,content_77d3a2a3d81fd40c,2026-03-25,571,101.861646,1,0.995544,high_visibility_low_position,review_ranking_opportunity


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*


### Intended use

The playbook is intended to help content teams prioritize which pages deserve human review first. It is a decision-support tool for ranking and content optimization workflows.

The output should help reviewers focus on pages with stronger observable search signals and possible ranking opportunities.

### Limits

The playbook does not establish that a page will improve after optimization. The model and baseline provide directional signals rather than causal evidence.

The validated grouped split showed an F1 score of 0.319, compared with 0.387 under the random split. This difference indicates that performance depends on the validation design and that generalization to unseen clients is more difficult.

The data also has incomplete GSC and GA4 coverage for some observations. Therefore, missing or unavailable signals should be considered during review.

The playbook should not be treated as a production automation system.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*


### Human review rules

Every recommended action requires human review before implementation.

The reviewer should:

1. Confirm that the page has sufficient search data.
2. Check whether the search visibility represents valuable or relevant queries.
3. Check the page's search intent and whether it matches the queries generating impressions.
4. Check whether the page is already improving or changing.
5. Consider whether the recommended action is appropriate for the page type.
6. Reject the recommendation when the underlying data is incomplete or misleading.

### No-go cases

The following should not be automated:

- Publishing or rewriting content automatically.
- Changing search intent without human review.
- Removing or redirecting a page automatically.
- Making decisions from incomplete GSC/GA4 data without review.
- Treating a model prediction as proof of future ranking improvement.
- Applying an action when the page's business or search context is unknown.
- Automatically changing titles, headings, internal links, or content based only on the score.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*


The playbook should be monitored periodically rather than assumed to remain stable.

Useful monitoring signals include:

- F1 score on a newly available validation window.
- Precision and recall for the improvement target.
- Changes in the distribution of action scores.
- Changes in GSC data availability.
- Changes in the number of pages receiving each reason code.
- Changes in false-positive and false-negative rates.

A model review or retraining cycle should be considered when:

- F1 declines materially on a recent held-out period.
- Error patterns change substantially.
- The distribution of the input features shifts.
- Data availability changes significantly.
- New content types or client groups are introduced.

Monitoring should trigger investigation and human review, not automatic model replacement.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*


The ranked queue is exported to `work/outputs/` so that the paper can trace its recommendations back to the notebook.

The exported queue contains the page identifier, observed search signals, action score, reason code, and recommended human-review action.

In [11]:
output_dir = "/content/work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = f"{output_dir}/content_action_playbook_queue.csv"

queue.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(queue))
print("Columns:", queue.columns.tolist())

Saved: /content/work/outputs/content_action_playbook_queue.csv
Rows: 9841378
Columns: ['rank', 'client_hash_id', 'content_hash_id', 'report_date', 'gsc_impressions', 'gsc_avg_position', 'gsc_clicks', 'action_score', 'reason_code', 'action']


In [12]:
display(
    pd.read_csv(output_path).head(10)
)

,rank,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_avg_position,gsc_clicks,action_score,reason_code,action
0,1,client_20259bd6705d81d4,content_1c14838b50c55c9f,2026-03-25,1190,134.111765,0,0.998694,high_visibility_low_position,review_ranking_opportunity
1,2,client_20259bd6705d81d4,content_f4bd56290b265782,2026-03-25,1146,105.913613,0,0.998492,high_visibility_low_position,review_ranking_opportunity
2,3,client_20259bd6705d81d4,content_b6259d89f57d5ebc,2026-03-25,986,96.248479,3,0.996996,high_visibility_low_position,review_ranking_opportunity
3,4,client_20259bd6705d81d4,content_6c67f84d8b90e7c5,2026-03-25,689,118.474601,0,0.996719,high_visibility_low_position,review_ranking_opportunity
4,5,client_20259bd6705d81d4,content_dbdef2d26fa09819,2026-03-25,667,117.287856,0,0.996541,high_visibility_low_position,review_ranking_opportunity
5,6,client_20259bd6705d81d4,content_4c8f60c8875f8cc6,2026-03-25,640,99.345313,4,0.996198,high_visibility_low_position,review_ranking_opportunity
6,7,client_20259bd6705d81d4,content_3896bdc4b2b82656,2026-03-25,617,121.458671,0,0.996122,high_visibility_low_position,review_ranking_opportunity
7,8,client_20259bd6705d81d4,content_4c82c93a3ee8cba8,2026-03-25,560,149.058929,0,0.995586,high_visibility_low_position,review_ranking_opportunity
8,9,client_23a62021009f63c4,content_6530fa9d297c46eb,2026-03-31,5364,89.848248,0,0.995552,high_visibility_low_position,review_ranking_opportunity
9,10,client_20259bd6705d81d4,content_77d3a2a3d81fd40c,2026-03-25,571,101.861646,1,0.995544,high_visibility_low_position,review_ranking_opportunity


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Archetype → action mapping

| Content archetype | Action | Human review |
|---|---|---|
| High visibility, weak position | Review ranking opportunity | Required |
| Low visibility, weak position | Gather more evidence before action | Required |
| High visibility, strong position | Monitor rather than change | Required |
| Low visibility, strong position | Low priority for this playbook | Required |

These mappings are decision-support categories rather than automatic production rules.

### Decay / refresh insight

Content age and freshness can be useful contextual signals when evaluating whether a ranking opportunity may be related to content decay or staleness. However, the current validated model does not establish that refreshing older content causes ranking improvement.

Therefore, freshness should be treated as a review signal rather than an automatic refresh trigger.

### Cost / value thinking

The highest-ranked pages should not automatically receive the most expensive intervention.

A practical review order is to prioritize pages where:

- search visibility is meaningful,
- the potential opportunity is plausible,
- the page is valuable enough to justify review effort,
- and the evidence quality is sufficient.

Low-confidence pages should receive lower-cost investigation before expensive content work.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.